Goal:
- Build an AI assistant that can answer underwriting questions.
- Uses RAG: retrieves similar policies and generates explanations.
- Maintains interpretability and context for decisions.
- Integrates with Groq API for LLM responses.

In [1]:
import pandas as pd

df = pd.read_csv("insurance_dataset.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               50000 non-null  int64  
 1   driver_type       50000 non-null  object 
 2   years_experience  50000 non-null  int64  
 3   income_band       50000 non-null  object 
 4   vehicle_age       50000 non-null  int64  
 5   vehicle_type      50000 non-null  object 
 6   vehicle_value     50000 non-null  float64
 7   airbags           50000 non-null  int64  
 8   tracking_device   50000 non-null  int64  
 9   region            50000 non-null  object 
 10  traffic_density   50000 non-null  object 
 11  policy_duration   50000 non-null  int64  
 12  annual_mileage    50000 non-null  int64  
 13  speeding_score    50000 non-null  object 
 14  previous_claims   50000 non-null  int64  
 15  fraud_flag        50000 non-null  int64  
 16  risk_score        50000 non-null  float6

### Preparing Knowledge Base (all policies, key features, model logic.) 

- We’ll convert each policy into a text chunk that can be embedded/retrieved.

In [3]:
# Select important features for KB
kb_features = ["age", "driver_type", "vehicle_type", "vehicle_value", "vehicle_age", "annual_mileage", "previous_claims", "risk_score", "airbags", "tracking_device", "region", "policy_duration"]

# Create text representation of each policy
def policy_to_text(row):
    return (
        f"Policyholder: age {row['age']}, driver_type {row['driver_type']}, "
        f"vehicle_type {row['vehicle_type']}, vehicle_value {row['vehicle_value']}, ({row['vehicle_age']} yrs old), "
        f"annual_mileage {row['annual_mileage']}, previous_claims {row['previous_claims']}, "
        f"safety: airbags {row['airbags']}, tracking_device {row['tracking_device']}, "
        f"region {row['region']}, policy_duration {row['policy_duration']} months, "
        f"risk_score {row['risk_score']:.2f}."
    )

df["policy_text"] = df.apply(policy_to_text, axis=1)

In [4]:
df["policy_text"].head(5)

0    Policyholder: age 59, driver_type private, veh...
1    Policyholder: age 49, driver_type taxi, vehicl...
2    Policyholder: age 35, driver_type private, veh...
3    Policyholder: age 63, driver_type taxi, vehicl...
4    Policyholder: age 28, driver_type taxi, vehicl...
Name: policy_text, dtype: object

### Vectorizing Policies
We'll use TF-IDF embeddings for similarity (works surprisingly well for tabular-to-text embeddings). Cosine similarity will measure “policy similarity” for RAG

Switch to Groq embeddings API for better results.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000) # avoids overfitting & too sparse vectors
policy_vectors = vectorizer.fit_transform(df["policy_text"])

In [6]:
policy_vectors.shape

(50000, 5000)

### Retrieval Function

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_policies(query_text, top_k=5):
    query_vec = vectorizer.transform([query_text])
    sims = cosine_similarity(query_vec, policy_vectors).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return df.iloc[top_idx][["policy_text", "risk_score", "claim_occurred"]], sims[top_idx]

In [10]:
query = "Private driver, 28 yrs, sedan, 5 yrs old, 18000 km/yr, 0 previous claims, Nairobi, 12 months policy"
similar_policies, scores = retrieve_similar_policies(query)
similar_policies

,policy_text,risk_score,claim_occurred
37359,"Policyholder: age 28, driver_type private, veh...",0.28,0
3195,"Policyholder: age 28, driver_type private, veh...",0.28,0
48650,"Policyholder: age 28, driver_type private, veh...",0.28,1
35484,"Policyholder: age 28, driver_type private, veh...",0.28,0
15515,"Policyholder: age 28, driver_type private, veh...",0.28,0


### Groq Setup

We’ll combine retrieved policies and user question into a prompt

In [34]:
import joblib

model = joblib.load("model.pkl")
feature_columns = joblib.load("feature_columns.pkl")

In [39]:
def calculate_premium(predicted_risk, vehicle_value,
                      expense_loading=0.3, profit_margin=0.1):
    """
    Calculate technical premium using expected loss approach
    """
    
    # Expected loss (pure premium)
    expected_loss = predicted_risk * vehicle_value
    
    # Add loadings
    premium = expected_loss * (1 + expense_loading + profit_margin)
    premium = max(premium, 5000)  # avoid unrealistically low premiums
    
    return expected_loss, premium

In [35]:
def predict_risk_from_query(query_dict):
    input_df = pd.DataFrame([query_dict])
    
    # Convert categoricals
    input_df = pd.get_dummies(input_df)
    
    # Align with training features
    input_df = input_df.reindex(columns=feature_columns, fill_value=0)
    
    return model.predict_proba(input_df)[:, 1][0]

In [18]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = ChatGroq(
    temperature=0.3,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

def ask_groq(prompt):
    response = client.invoke([
        {"role": "system", "content": "You are an expert insurance underwriter."},
        {"role": "user", "content": prompt}
    ])
    
    return response.content

In [42]:
def build_prompt(user_question, retrieved_policies, predicted_risk, premium):
    
    context = "\n".join(retrieved_policies["policy_text"].tolist())
    
    prompt = f"""
You are an AI insurance underwriting assistant.

PREDICTED RISK SCORE: {predicted_risk:.2f}
RECOMMENDED PREMIUM: {premium:.0f}

SIMILAR POLICIES:
{context}

QUESTION:
{user_question}

INSTRUCTIONS:
- Provide underwriting decision (Approve / Conditional / Decline)
- Explain reasoning using risk score and similar cases
- Justify the recommended premium
- Highlight key risk drivers

ANSWER:
"""
    return prompt

### Testing the RAG Assistant

In [40]:
query_features = {
    "age": 30,
    "driver_type": "private",
    "vehicle_age": 3,
    "vehicle_type": "sedan",
    "vehicle_value": 1200000,
    "annual_mileage": 15000,
    "previous_claims": 0,
    "airbags": 4,
    "tracking_device": 1,
    "region": "Nairobi",
    "policy_duration": 12,
    "speeding_score": "medium"
}

In [41]:
vehicle_value = query_features["vehicle_value"]

expected_loss, recommended_premium = calculate_premium(predicted_risk, vehicle_value)

print(f"Expected Loss: {expected_loss:.2f}")
print(f"Recommended Premium: {recommended_premium:.2f}")

Expected Loss: 396427.93
Recommended Premium: 554999.11


In [43]:
import requests

# Example question
user_question = "What is the risk profile for a 30-year-old private driver with a 3-year-old sedan in Nairobi, no previous claims?"

# Step 1: Retrieve similar policies
retrieved_policies, sims = retrieve_similar_policies(user_question)

# Step 2: Predict risk
predicted_risk = predict_risk_from_query(query_features)

# Step 3: Pricing
expected_loss, premium = calculate_premium(predicted_risk, query_features["vehicle_value"])

# Step 4: Build prompt
prompt = build_prompt(user_question, retrieved_policies, predicted_risk, premium)

# Step 5: Ask LLM
answer = ask_groq(prompt)

print(answer)

**Underwriting Decision:** Approve

**Reasoning:**
The predicted risk score of 0.35 is relatively low, indicating a moderate risk profile. Upon reviewing similar policies, we notice that policyholders with similar characteristics (age 30, private driver, sedan, no previous claims, and safety features such as airbags and tracking devices) have a risk score of 0.30. Although the vehicle in question is 3 years old, which may slightly increase the risk due to potential depreciation and wear and tear, the overall profile remains favorable.

The similar cases suggest that private drivers in Nairobi with sedans and no previous claims tend to have a lower risk score. The presence of safety features such as airbags and tracking devices also contributes to a lower risk profile. However, the slightly higher annual mileage and older vehicle age may contribute to the slightly higher risk score of 0.35 compared to the similar cases.

**Recommended Premium:**
The recommended premium of 592,495 is jus

In [44]:
risk_avg = retrieved_policies["risk_score"].mean()
claim_rate = retrieved_policies["claim_occurred"].mean()

print(f"Average risk_score of similar policies: {risk_avg:.2f}")
print(f"Claim rate among similar policies: {claim_rate:.2%}")

Average risk_score of similar policies: 0.30
Claim rate among similar policies: 0.00%
